# Trabajos Bases de Datos a Gran Escala, 2026-2027

En las siguientes transparencias se incluyen los trabajos propuestos para la asignatura. Cada trabajo incluye el estudio de una base de datos no vista en clase y la importación de los datos de Stackoverflow.

**El trabajo será INDIVIDUAL**.

Más de un alumno podrá elegir el mismo trabajo, pero como máximo un trabajo podrá ser elegido por dos alumnos. Se entregará una memoria (y código, si se ha generado) en
una tarea abierta a tal fin.

La **fecha de entrega será el día del examen de la asignatura**.

Para apuntarse a un trabajo el alumno tendrá que:

1. Conectarse al servidor. Esto se puede hacer en alguno de los *notebooks* de SQL, se puede abrir la terminal en Google Colab y usar el cliente de MySQL.
   Usuario: alumno2627,
   pass: (se enviará aparte en un mensaje en el aula virtual)

   ```bash
   $ mysql -hdsevilla-proxy.inf.um.es -P3307 --default-character-set=utf8mb4 \
       --init-command="SET NAMES utf8mb4 COLLATE utf8mb4_unicode_ci" \
       -ualumno2627 -p<password> --protocol=tcp trabajos2627
   ```
   La intercalación es `utf8mb4_unicode_ci`, la misma con la que se crearon las tablas de este servicio (ver `creatrabajos.sql`), y no la `utf8mb4_0900_as_ci` de la sesión 2: se usa para que el DNI se compare sin distinguir mayúsculas de minúsculas, como en la normalización del disparador. Esto también se puede hacer en los *notebooks* de SQL, cambiando la cadena de conexión de la celda del Notebook de la sesión 2 a:

   ```sql
   %%sql
   mysql+pymysql://alumno2627:<password>@dsevilla-proxy.inf.um.es:3307/?charset=utf8mb4&collation=utf8mb4_unicode_ci
   ```

2. Añadir una entrada a la tabla `asignacion_trabajos` con su DNI y el identificador del trabajo elegido:

   ```sql
   INSERT INTO asignacion_trabajos VALUES ('01234567L', 'T07');
   ```

- Sólo se guardan esos dos datos: el nombre no se almacena en el servidor
- Se valida el formato del DNI o NIE (`01234567L`, `X1234567L`) **y su letra de control**, que es el error más habitual al teclearlo
- Da igual escribirlo en mayúsculas o minúsculas, y con guion o sin él: el sistema lo normaliza antes de guardarlo. Lo mismo con el identificador del trabajo (`t07` vale)
- Sólo se permite un trabajo por alumno
- Máximo 2 alumnos por trabajo: si ya hubiera dos asignados, el sistema rechaza el alta

3. Consultas útiles:

```sql
mysql> USE trabajos2627;
Database changed

mysql> SELECT id, titulo FROM trabajos;
+-----+-----------------------+
| id  | titulo                |
+-----+-----------------------+
| Txx | Título                |
| ... | .................     |
+-----+-----------------------+

mysql> SELECT * FROM asignados;
+-----+-----------------------+------------+
| id  | titulo                | nasignados |
+-----+-----------------------+------------+
| Txx | Título                |          0 |
| ... | ................      |          0 |
+-----+-----------------------+------------+

mysql> SHOW TABLES;
+-----------------------+
| Tables_in_trabajos2627 |
+-----------------------+
| asignacion_trabajos   |
| asignados             |
| trabajos              |
+-----------------------+
```

`asignacion_trabajos` no se puede consultar: sólo se puede insertar en ella. Así nadie ve los DNI de los demás. Para saber qué está cogido, usa la vista `asignados`.

4. Si te equivocas al elegir, contacta con el profesor para el cambio. No es posible reasignarse directamente si ya tienes un trabajo asignado.

Notas sobre consultas RQ: en cada trabajo se pide reproducir (cuando aplique) las consultas RQ1–RQ4 de la sesión 2 usando el stack de la tecnología elegida (conectores/engines adecuados). Asegúrate de documentar los supuestos, índices/configuración y límites de cada motor al ejecutar dichas consultas.

## T01 -- CockroachDB

- https://www.cockroachlabs.com/product/
- Pasos de instalación/arranque (cluster local con `cockroach start-single-node` o 3 nodos en Docker)
- Descripción: base de datos SQL distribuida, transacciones ACID multi-región, compatibilidad PostgreSQL, consenso Raft.
- Modelado y características: replicación, particionamiento por rangos, índices, transacciones, funciones.
- Importar los datos de Stackoverflow (CSV/Parquet vía `IMPORT INTO` o ingestión por `cockroach sql`/`COPY`)
- Redistribución óptima (particionado por rangos/geografía, índices compuestos)
- Reproducir RQ1–RQ4 de sesión 2
- Benchmarks comparativos con una BD de la asignatura
- Artículo/whitepaper: https://www.cockroachlabs.com/blog/consistent-replication/ (Raft)

## T02 -- Valkey: almacén clave-valor en memoria (bifurcación de Redis)

- https://valkey.io/ | https://redis.io/
- Pasos de instalación (contenedor único y clúster de 3 nodos con `valkey-cli --cluster create`)
- Descripción: almacén clave-valor en memoria con estructuras de datos (cadenas, *hashes*, listas, conjuntos, conjuntos ordenados, *streams*); persistencia con RDB y AOF; replicación y clúster con 16384 *slots*
- Historia de licencias como parte del análisis: Redis pasa de BSD a RSAL/SSPL en 2024, la Linux Foundation bifurca Valkey desde la 7.2.4 y Redis 8 vuelve al código abierto con AGPLv3 en 2025. Qué implica cada licencia para quien lo ofrece como servicio
- Importar Stackoverflow eligiendo la estructura según la consulta: `HASH` por post, conjuntos por etiqueta, conjuntos ordenados para rankings por puntuación o fecha
- Implementar RQ1–RQ4 construyendo a mano los índices que haría falta mantener, y comparar el diseño con las tablas dirigidas por consultas de Cassandra
- Medir el coste de la persistencia (sin ella, RDB, AOF con `everysec` y con `always`) y el de la replicación
- Explicar la migración atómica de *slots* de Valkey 9 frente a la migración clave a clave anterior
- Artículos: Nishtala et al., «Scaling Memcache at Facebook», NSDI 2013; Ousterhout et al., «The RAMCloud Storage System», ACM TOCS 33(3), 2015
- Fuentes: [Redis vuelve al código abierto con AGPL](https://www.theregister.com/software/2025/05/01/redis-returns-to-open-source-with-agpl-license/757582); [Valkey 9.0](https://valkey.io/blog/introducing-valkey-9/)

## T03 -- Elasticsearch y ParadeDB: búsqueda de texto con BM25

- https://www.elastic.co/elasticsearch | https://www.paradedb.com/
- Pasos de instalación: Elasticsearch en contenedor único y clúster de 3 nodos; ParadeDB en contenedor (`paradedb/paradedb`, PostgreSQL con la extensión `pg_search`)
- Descripción de Elasticsearch: índice invertido sobre Lucene, analizadores por idioma, *mappings*, fragmentos y réplicas, agregaciones, relevancia con BM25; licencia AGPLv3 (junto a SSPL y ELv2) desde 2024 y bifurcación OpenSearch
- Descripción de ParadeDB: búsqueda BM25 dentro de PostgreSQL con Tantivy (alternativa a Lucene en Rust), sin sincronizar un segundo sistema; licencia AGPLv3
- Importar Stackoverflow en los dos: `Title` y `Body` con analizador de español, `Tags` como palabras clave
- Búsquedas de texto: frases, *fuzzy*, resaltado y facetas por etiqueta; comparar resultados y tiempos entre los dos y con el `FULLTEXT` de MySQL de la sesión 2
- Ejecutar RQ1–RQ4 con agregaciones de Elasticsearch y con SQL en ParadeDB; discutir qué pasa con las reuniones (RQ4) cuando los datos no están desnormalizados
- Discutir la decisión de fondo: un motor de búsqueda aparte frente a la búsqueda dentro de la base de datos transaccional (consistencia, operación, rendimiento)
- Artículos: Zobel y Moffat, «Inverted Files for Text Search Engines», ACM Computing Surveys 38(2), 2006; Robertson y Zaragoza, «The Probabilistic Relevance Framework: BM25 and Beyond», Foundations and Trends in Information Retrieval 3(4), 2009
- Fuentes: [Elasticsearch vuelve a ser código abierto](https://www.elastic.co/blog/elasticsearch-is-open-source-again); [repositorio de ParadeDB](https://github.com/paradedb/paradedb); [imagen Docker de ParadeDB](https://hub.docker.com/r/paradedb/paradedb)

## T04 -- Couchbase y SQL++

- https://www.couchbase.com/ (Community Edition gratuita, hasta 5 nodos)
- Pasos de instalación (contenedor único y clúster de 3 nodos); servicios de datos, índices, consultas, búsqueda y analítica, que se pueden colocar en nodos distintos
- Descripción: base de datos documental con caché clave-valor integrada; *buckets*, *scopes* y colecciones; vBuckets para repartir los datos; lenguaje SQL++ (antes N1QL) sobre JSON; índices secundarios globales; índices vectoriales desde la versión 8.0
- Importar Stackoverflow con `cbimport`, decidiendo qué se agrega y qué se referencia (como en la sesión de MongoDB)
- Ejecutar RQ1–RQ4 en SQL++ con `JOIN`, `NEST` y `UNNEST`; mostrar con `EXPLAIN` qué índices se usan y crear los que falten
- Comparar con MongoDB: SQL++ frente al pipeline de agregación, y acceso por clave desde la caché frente a consulta
- Explicar la separación de servicios (escalado multidimensional) y cuándo compensa
- Artículos: Ong, Papakonstantinou y Vernoux, «The SQL++ Query Language: Configurable, Unifying and Semi-structured», arXiv:1405.3631, 2014; Borkar et al., «Have Your Data and Query It Too: From Key-Value Caching to Big Data Management», SIGMOD 2016
- Fuentes: [ediciones de Couchbase Server](https://docs.couchbase.com/server/current/introduction/editions.html); [novedades de la versión 8.0](https://docs.couchbase.com/server/current/introduction/whats-new.html)

## T05 -- FoundationDB

- https://www.foundationdb.org/
- Pasos de instalación (binarios oficiales / Docker); administración básica de clusters
- Descripción: base de datos clave-valor distribuida con transacciones ACID y capas (SQL, Document, Graph) sobre un KV ordenado.
- Modelo y características: transacciones multi-clave, MVCC, consistencia fuerte, punto de control, replicación automática.
- Cargar Stackoverflow en una capa adecuada (p. ej., Record Layer o capa Document); diseño de esquemas/índices sobre la capa elegida
- Consultas RQ1–RQ4 usando la capa seleccionada (p. ej., Record Layer + consultas con índices)
- Benchmarks/latencias frente a otras soluciones NoSQL de la lista
- Research/tech: FoundationDB Record Layer (Apple) https://github.com/FoundationDB/fdb-record-layer | Arquitectura FDB https://www.foundationdb.org/files/fdb-paper.pdf

## T06 -- InfluxDB 3: series temporales sobre Arrow, DataFusion y Parquet

- https://www.influxdata.com/ | https://github.com/influxdata/influxdb (InfluxDB 3 Core, MIT/Apache 2)
- Pasos de instalación (contenedor de InfluxDB 3 Core); escritura con *line protocol* y consulta con SQL o InfluxQL
- Descripción: base de datos de series temporales reescrita sobre la pila FDAP (Flight, DataFusion, Arrow, Parquet); separación entre ingesta en memoria y ficheros Parquet persistidos; qué cambia respecto a InfluxDB 1 y 2 (TSM, Flux)
- Modelar Stackoverflow como eventos en el tiempo: publicación de preguntas y respuestas, votos y comentarios, con etiquetas y usuario como *tags*
- Consultas propias de series temporales: actividad por etiqueta y hora, tiempo hasta la primera respuesta, ventanas y submuestreo
- Discutir qué parte de RQ1–RQ4 encaja en una base de datos de series temporales y cuál no (reuniones entre usuarios y posts)
- Explicar la cardinalidad de las series y por qué era el límite de las versiones anteriores
- Comparar con MongoDB (colecciones de series temporales) o con ClickHouse sobre las mismas consultas
- Artículos: Pelkonen et al., «Gorilla: A Fast, Scalable, In-Memory Time Series Database», PVLDB 8(12), 2015; Lamb et al., «Apache Arrow DataFusion: A Fast, Embeddable, Modular Analytic Query Engine», SIGMOD 2024
- Fuentes: [InfluxDB 3 como código abierto](https://www.infoq.com/news/2025/04/influxdb3-open-source)

## T07 -- ArangoDB: multimodelo documento + grafo

- https://arango.ai/ (desde la 3.12, código BSL 1.1; usar la Community Edition, gratuita para uso no comercial y hasta 100 GiB de datos)
- Pasos de instalación (contenedor único y clúster con coordinadores, *DB-servers* y *agents*)
- Descripción: base de datos multimodelo con documentos JSON, grafos y clave-valor sobre el mismo motor (RocksDB); lenguaje AQL; *SmartGraphs* y fragmentación
- Importar Stackoverflow con `arangoimport`: colecciones de documentos para usuarios y posts, colecciones de aristas para autoría, respuesta y etiquetado
- Ejecutar RQ1–RQ4 en AQL, mezclando consultas documentales y recorridos, y comparar con MongoDB (parte documental) y con Neo4j (parte de grafo)
- Discutir qué se gana y qué se pierde con un único motor multimodelo frente a la persistencia políglota
- Explicar el efecto de la licencia BSL y del límite de la Community Edition sobre su adopción
- Artículos: Angles et al., «Foundations of Modern Query Languages for Graph Databases», ACM Computing Surveys 50(5), 2017; Lu y Holubová, «Multi-model Databases: A New Journey to Handle the Variety of Data», ACM Computing Surveys 52(3), 2019
- Fuentes: [cambio de licencia de ArangoDB](https://arango.ai/blog/evolving-arangodbs-licensing-model-for-a-sustainable-future/)

## T08 -- Apache Druid: analítica en tiempo real

- https://druid.apache.org/
- Pasos de instalación (configuración `micro-quickstart` o Docker Compose); papel de cada proceso (*coordinator*, *overlord*, *broker*, *historical*, *middle manager*)
- Descripción: almacén analítico columnar orientado a eventos con marca de tiempo; segmentos inmutables particionados por tiempo, *rollup* en la ingesta, índices de mapas de bits, almacenamiento profundo
- Importar Stackoverflow por lotes (ingesta nativa desde CSV o Parquet), eligiendo la columna de tiempo y la granularidad de los segmentos
- Medir el efecto del *rollup* en tamaño y en precisión de las consultas
- Ejecutar con Druid SQL las consultas de RQ1–RQ4 que encajen; justificar por qué las reuniones grandes (RQ4) no son su caso de uso
- Comparar con ClickHouse sobre las mismas agregaciones
- Artículos: Yang et al., «Druid: A Real-time Analytical Data Store», SIGMOD 2014

## T09 -- ClickHouse: SQL columnar para analítica

- https://clickhouse.com/
- Pasos de instalación (contenedor único o `clickhouse local` sobre ficheros)
- Descripción: base de datos columnar; motores de tabla de la familia MergeTree, clave de ordenación e índice disperso, *data skipping indexes*, compresión por columna, ejecución vectorizada
- Importar Stackoverflow directamente desde los Parquet del curso
- Ejecutar RQ1–RQ4 y optimizarlas: elegir `ORDER BY` y `PARTITION BY`, proyecciones y vistas materializadas incrementales; mostrar con `EXPLAIN` qué gránulos se leen
- Analizar el coste de las reuniones (RQ4) y los algoritmos de reunión disponibles
- Comparar con DuckDB y MySQL en tiempo y en espacio en disco
- Artículos: Schulze et al., «ClickHouse - Lightning Fast Analytics for Everyone», PVLDB 17(12), 2024; O'Neil et al., «The Log-Structured Merge-Tree (LSM-Tree)», Acta Informatica 33(4), 1996

## T10 -- SurrealDB: multimodelo con un único lenguaje

- https://surrealdb.com/
- Pasos de instalación (binario o contenedor); modo en memoria, con RocksDB/SurrealKV y distribuido sobre TiKV
- Descripción: base de datos multimodelo (documentos, grafos, clave-valor, series temporales, vectores y búsqueda de texto) con el lenguaje SurrealQL; registros con identificadores enlazables y aristas con `RELATE`; novedades de la versión 3.0
- Importar Stackoverflow modelando usuarios, preguntas y respuestas como registros y las relaciones como aristas
- Ejecutar RQ1–RQ4 en SurrealQL combinando consultas de documentos y recorridos, y mostrar qué índices se usan
- Comparar con ArangoDB o con la pareja MongoDB + Neo4j: qué ahorra un único lenguaje y a qué precio en rendimiento
- Artículos: Lu y Holubová, «Multi-model Databases: A New Journey to Handle the Variety of Data», ACM Computing Surveys 52(3), 2019
- Fuentes: [SurrealDB 3.0](https://surrealdb.com/releases/3.0)

## T11 -- RavenDB: documentos con índices construidos en segundo plano

- https://ravendb.net/ (licencia de desarrollo o *Community* gratuita)
- Pasos de instalación (contenedor único y clúster de 3 nodos)
- Descripción: base de datos documental transaccional (ACID); índices *map* y *map-reduce* que se actualizan de forma asíncrona, índices automáticos creados por el optimizador, lenguaje RQL, replicación maestro-maestro
- Importar Stackoverflow como documentos, decidiendo qué se agrega y qué se referencia (como en la sesión de MongoDB)
- Implementar RQ1–RQ4 con índices *map-reduce*; explicar las consultas «obsoletas» (*stale*) y cómo esperar a que el índice esté al día
- Comparar con MongoDB: índices síncronos frente a asíncronos, pipeline de agregación frente a *map-reduce* materializado
- Artículos: Eini, «Inside RavenDB» (libro, disponible gratuitamente en la web de RavenDB); Dean y Ghemawat, «MapReduce: Simplified Data Processing on Large Clusters», OSDI 2004

## T12 -- ScyllaDB: Cassandra reescrita con un fragmento por núcleo

- https://www.scylladb.com/ (desde 2025, licencia *source-available* con nivel gratuito)
- Pasos de instalación (contenedor único y clúster de 3 nodos)
- Descripción: base de datos compatible con Cassandra (CQL) escrita en C++ sobre Seastar; arquitectura *shard-per-core* sin memoria compartida; *tablets* gestionadas con Raft en lugar de los *vnodes* del anillo
- Reutilizar el modelo dirigido por consultas de la sesión de Cassandra para cargar Stackoverflow e implementar RQ1–RQ4
- Comparar con Cassandra con la misma carga: latencias de cola (p99), rendimiento y tiempo de añadir un nodo
- Explicar las *tablets* frente a los *vnodes* y medir el reequilibrado al crecer el clúster
- Discutir el cambio de licencia de 2024 y el fin de la versión AGPL
- Artículos: Lakshman y Malik, «Cassandra: A Decentralized Structured Storage System», ACM SIGOPS OSR 44(2), 2010; DeCandia et al., «Dynamo: Amazon's Highly Available Key-value Store», SOSP 2007
- Fuentes: [licencia *source-available* de ScyllaDB](https://www.scylladb.com/source-available-faq/); [ScyllaDB 6.0 y las *tablets*](https://www.scylladb.com/2024/06/12/introducing-scylladb-6-0-with-tablets-and-strongly-consistent-topology-updates/)

## T13 -- Apache Geode: rejilla de datos en memoria

- https://geode.apache.org/ (versión 2.0, de diciembre de 2025)
- Pasos de instalación con `gfsh`: localizador, servidores y regiones replicadas y particionadas
- Descripción: rejilla de datos en memoria (origen: GemFire); regiones, particionado con copias redundantes, persistencia opcional, consultas OQL, funciones ejecutadas junto a los datos, notificación continua de cambios
- Importar Stackoverflow en regiones particionadas y colocar juntos los datos que se consultan juntos (*colocation*)
- Implementar RQ1–RQ4 con OQL y con funciones distribuidas; comparar los tiempos
- Construir un API REST sobre los datos
- Medir la recuperación ante la caída de un servidor y el efecto del número de copias redundantes
- Artículos: Ousterhout et al., «The RAMCloud Storage System», ACM TOCS 33(3), 2015; Stonebraker et al., «The End of an Architectural Era (It's Time for a Complete Rewrite)», VLDB 2007
- Fuentes: [anuncio de Apache Geode 2.0](https://news.apache.org/foundation/entry/the-apache-software-foundation-announces-apache-geode-2-0)

## T14 -- DuckLake y MotherDuck: *lakehouse* con catálogo SQL y ejecución híbrida

- https://ducklake.select/ | https://motherduck.com/ (plan Lite gratuito, sin tarjeta)
- DuckDB ya se ha visto en la sesión 1: el trabajo trata sólo de lo que se construye encima
- Pasos de instalación: extensión `ducklake` de DuckDB con el catálogo en SQLite o PostgreSQL y los datos en ficheros Parquet locales o en almacenamiento de objetos; cuenta de MotherDuck
- Descripción de DuckLake: formato de *lakehouse* que guarda los metadatos en una base de datos SQL en lugar de en miles de ficheros JSON/Avro; transacciones, instantáneas, *time travel*, evolución del esquema, *data inlining* de escrituras pequeñas
- Cargar Stackoverflow como tablas DuckLake, hacer altas y modificaciones y consultar versiones anteriores
- Comparar el diseño con el catálogo por ficheros de Iceberg y Delta Lake: cuántas lecturas hacen falta para planificar una consulta y cómo se resuelven las escrituras concurrentes
- Descripción de MotherDuck: ejecución híbrida, con parte del plan en el cliente y parte en la nube; bases de datos compartidas
- Ejecutar RQ1–RQ4 en local, en MotherDuck y en modo híbrido; explicar con `EXPLAIN` qué operadores se ejecutan en cada lado y cuánto se transfiere
- Artículos: Atwal et al., «MotherDuck: DuckDB in the Cloud and in the Client», CIDR 2024; Raasveldt y Mühleisen, «DuckDB: an Embeddable Analytical Database», SIGMOD 2019; Armbrust et al., «Lakehouse: A New Generation of Open Platforms that Unify Data Warehousing and Advanced Analytics», CIDR 2021
- Fuentes: [DuckLake 1.0](https://ducklake.select/2026/04/13/ducklake-10/); [precios de MotherDuck](https://motherduck.com/product/pricing/); [artículo de MotherDuck en CIDR 2024](https://www.cidrdb.org/cidr2024/papers/p46-atwal.pdf)

## T15 -- Materialize

- https://materialize.com/
- Pasos de instalación (Docker o cloud trial); creación de fuentes, vistas materializadas, y sinks.
- Descripción: base de datos de streaming SQL con vistas materializadas incrementales (arriba de Kafka, Postgres, HTTP, etc.).
- Ingesta de Stackoverflow como streams o batch; creación de vistas para RQ1–RQ4 con actualizaciones en tiempo real.
- Explicar consistencia, “reclustering”, índices, TTL; diferenciar de Flink/Spark streaming.
- Comparativa con motores de streaming SQL (Flink SQL, ksqlDB).
- Research/Tech: Differential Dataflow/Timely Dataflow (papers), docs https://materialize.com/docs/.

## T16 -- CedarDB (Umbra): compilación de consultas y paralelismo por *morsels*

- https://cedardb.com/ (Community Edition gratuita, hasta 64 GiB de datos)
- Pasos de instalación (`docker pull cedardb/cedardb`); acceso con `psql` o SQLAlchemy por el protocolo de PostgreSQL
- Descripción: sistema relacional en disco con rendimiento de sistema en memoria; compilación de consultas a código máquina, procesamiento por *morsels*, índices basados en Adaptive Radix Tree, almacenamiento híbrido por filas y columnas
- Cargar Stackoverflow desde los Parquet o CSV del curso, con claves, restricciones y estadísticas
- Ejecutar RQ1–RQ4 con el SQL de la sesión 2 y comparar planes y tiempos con MySQL y DuckDB; explicar de dónde sale la diferencia, no sólo medirla
- Centrar el análisis en RQ4 (autorreuniones de `Posts`), donde se nota la generación de código y el paralelismo por *morsels*
- El binario es cerrado: contrastar lo que prometen los artículos con lo que se mide
- Artículos: Neumann, «Efficiently Compiling Efficient Query Plans for Modern Hardware», PVLDB 4(9), 2011; Leis, Boncz, Kemper y Neumann, «Morsel-Driven Parallelism: A NUMA-Aware Query Evaluation Framework for the Many-Core Age», SIGMOD 2014; Neumann y Freitag, «Umbra: A Disk-Based System with In-Memory Performance», CIDR 2020
- Fuentes: [CedarDB Community Edition](https://cedardb.com/docs/community_edition/)

## T17 -- FalkorDB: grafos como matrices dispersas (GraphBLAS)

- https://www.falkordb.com/
- Pasos de instalación (`docker run falkordb/falkordb`); acceso con `redis-cli` o los clientes de Python
- Descripción: base de datos de grafos de propiedades, sucesora de RedisGraph, que representa la adyacencia como matrices dispersas y traduce los patrones Cypher a álgebra lineal con GraphBLAS
- Modelar Stackoverflow como grafo (usuarios, preguntas, respuestas, etiquetas y sus relaciones) y cargarlo desde los CSV o Parquet del curso
- Ejecutar RQ1–RQ4 en Cypher y comparar con Neo4j sobre el mismo grafo: planes, tiempos y memoria
- Analizar RQ4 (usuarios que se responden mutuamente) como producto de matrices: qué operación de GraphBLAS ejecuta el patrón y cuánto ocupan los resultados intermedios
- Discutir cuándo gana la representación matricial frente a las listas de adyacencia de Neo4j y cuándo no (recorridos profundos, grafos muy cambiantes, propiedades en las aristas)
- Artículos: Davis, «Algorithm 1000: SuiteSparse:GraphBLAS: Graph Algorithms in the Language of Sparse Linear Algebra», ACM TOMS 45(4), 2019; Kepner et al., «Mathematical Foundations of the GraphBLAS», IEEE HPEC 2016; Cailliau et al., «RedisGraph GraphBLAS Enabled Graph Database», IEEE IPDPSW 2019

## T18 -- TiDB: SQL distribuido y HTAP

- https://www.pingcap.com/ | https://github.com/pingcap/tidb
- Pasos de instalación (`tiup playground` o Docker Compose) con TiDB, TiKV, PD y TiFlash
- Descripción: base de datos SQL distribuida compatible con MySQL; almacenamiento clave-valor replicado con Raft (TiKV), transacciones al estilo Percolator y réplicas columnares (TiFlash) para consultas analíticas sobre los mismos datos
- Cargar Stackoverflow con el mismo esquema MySQL de la sesión 2 (`LOAD DATA` o TiDB Lightning)
- Ejecutar RQ1–RQ4 con y sin réplica en TiFlash; mostrar con `EXPLAIN ANALYZE` cuándo el optimizador elige cada almacenamiento
- Mezclar carga transaccional (altas de posts y votos) con las consultas analíticas y medir la interferencia entre ambas, que es el problema que HTAP intenta resolver
- Comparar con MySQL y con CockroachDB en particionado por rangos, consistencia y coste de las transacciones distribuidas
- Artículos: Huang et al., «TiDB: A Raft-based HTAP Database», PVLDB 13(12), 2020; Ongaro y Ousterhout, «In Search of an Understandable Consensus Algorithm», USENIX ATC 2014; Peng y Dabek, «Large-scale Incremental Processing Using Distributed Transactions and Notifications», OSDI 2010

## T19 -- Qdrant: base de datos vectorial y búsqueda aproximada (HNSW)

- https://qdrant.tech/
- Pasos de instalación (contenedor único); colecciones, índices HNSW y *payload*
- Descripción: almacenamiento de vectores densos y dispersos, métricas de distancia, índices de vecinos aproximados (ANN), filtrado por *payload*, cuantización y fragmentación de la colección
- Calcular los vectores de las preguntas de Stackoverflow (`Title` + `Body`) con un modelo de *embeddings* multilingüe (p. ej. `intfloat/multilingual-e5-small`): el corpus está en español y un modelo sólo en inglés da peores resultados, hay que medirlo
- Tarea evaluable 1: dada una pregunta nueva, recuperar las preguntas más parecidas y proponer su respuesta aceptada. Las preguntas con `AcceptedAnswerId` sirven de verdad de referencia: reservar una parte y medir Recall@k y MRR
- Tarea evaluable 2: predecir las `Tags` de una pregunta por votación de sus k vecinos y medir precisión y cobertura frente a las reales
- Línea base léxica: un índice `FULLTEXT` de MySQL sobre `Title` y `Body` (hay que crearlo; el de la sesión 2 es sobre `Tags`) con las mismas métricas
- Compromisos del índice: `m`, `ef_construct` y `ef` frente a *recall*, latencia y memoria, midiendo contra la búsqueda exacta; cuantización escalar, binaria y por producto; búsqueda híbrida (vectores densos + dispersos)
- Búsqueda filtrada: medir cómo cambia el *recall* al filtrar por etiqueta o fecha cada vez más selectivas, y explicar cómo lo resuelve Qdrant sobre el grafo HNSW
- Comparar con los índices vectoriales de MongoDB y Neo4j de los boletines: base de datos específica frente a capacidad añadida a una base de datos general
- Sobre RQ1–RQ4: son agregaciones y reuniones, no búsquedas por similitud. Discutir qué parte se puede hacer con filtros sobre el *payload* y por qué esta base de datos no es la herramienta adecuada para esas consultas
- Artículos: Malkov y Yashunin, «Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs», IEEE TPAMI 42(4), 2020; Jégou, Douze y Schmid, «Product Quantization for Nearest Neighbor Search», IEEE TPAMI 33(1), 2011; Gollapudi et al., «Filtered-DiskANN: Graph Algorithms for Approximate Nearest Neighbor Search with Filters», WWW 2023

## T20 -- Milvus: arquitectura de una base de datos vectorial distribuida

- https://milvus.io/
- Pasos de instalación: Milvus Lite (`pip install pymilvus`) para empezar y Milvus Standalone en Docker Compose (con etcd y MinIO) para las pruebas
- Descripción: base de datos vectorial nativa en la nube; separación de almacenamiento y cómputo, el registro (*log*) como columna vertebral, segmentos, nodos de consulta y de indexación, niveles de consistencia
- Calcular los vectores de las preguntas de Stackoverflow (`Title` + `Body`) con un modelo de *embeddings* multilingüe (p. ej. `intfloat/multilingual-e5-small`)
- Tarea evaluable: dada una pregunta nueva, recuperar las preguntas más parecidas y proponer su respuesta aceptada; usar las preguntas con `AcceptedAnswerId` como verdad de referencia y medir Recall@k y MRR
- Comparar tipos de índice sobre los mismos datos: `FLAT` (exacto), `IVF_FLAT`, `IVF_PQ`, `HNSW` y `DISKANN`; tiempo de construcción, memoria, latencia y *recall* frente a la búsqueda exacta
- Explicar cómo la arquitectura afecta a la frescura de los datos: cuándo es visible un vector recién insertado según el nivel de consistencia elegido
- Comparar con los índices vectoriales de MongoDB y Neo4j de los boletines, y con un índice léxico (`FULLTEXT` de MySQL sobre `Title` y `Body`) como línea base
- Sobre RQ1–RQ4: discutir qué parte se puede expresar con filtros escalares y por qué no es la herramienta adecuada para esas consultas
- Artículos: Wang et al., «Milvus: A Purpose-Built Vector Data Management System», SIGMOD 2021; Guo et al., «Manu: A Cloud Native Vector Database Management System», PVLDB 15(12), 2022; Subramanya et al., «DiskANN: Fast Accurate Billion-point Nearest Neighbor Search on a Single Node», NeurIPS 2019; Pan, Wang y Li, «Survey of Vector Database Management Systems», VLDB Journal 33, 2024

## T21 -- FerretDB y DocumentDB: el protocolo de MongoDB sobre PostgreSQL

- https://www.ferretdb.com/ | https://github.com/FerretDB/documentdb
- Pasos de instalación (contenedor de FerretDB con PostgreSQL y la extensión DocumentDB); acceso con `mongosh` y con los clientes de MongoDB
- Descripción: FerretDB traduce el protocolo de MongoDB a SQL; DocumentDB (de Microsoft, ahora en la Linux Foundation) es la extensión de PostgreSQL que almacena y consulta BSON
- Importar Stackoverflow con `mongoimport`, igual que en la sesión de MongoDB, y ejecutar sin cambios las consultas de la sesión y RQ1–RQ4 con el pipeline de agregación
- Mirar por debajo: cómo se guardan los documentos en las tablas de PostgreSQL, qué índices se crean y qué plan de PostgreSQL ejecuta cada consulta (`EXPLAIN`)
- Documentar las incompatibilidades encontradas y medir el rendimiento frente a MongoDB
- Discutir qué aporta tener los documentos en PostgreSQL (transacciones, reuniones con tablas relacionales, operación conocida) y el contexto de la demanda de MongoDB contra FerretDB en 2025
- Artículos: Chasseur, Li y Patel, «Enabling JSON Document Stores in Relational Systems», WebDB 2013; Liu et al., «Closing the Functional and Performance Gap between SQL and NoSQL», SIGMOD 2016
- Fuentes: [repositorio de FerretDB](https://github.com/FerretDB/FerretDB); [Pavlo, «Databases in 2025: A Year in Review»](https://www.cs.cmu.edu/~pavlo/blog/2026/01/2025-databases-retrospective.html)

## T22 -- Citus: PostgreSQL distribuido como extensión

- https://www.citusdata.com/ | https://github.com/citusdata/citus
- Pasos de instalación (Docker Compose con un coordinador y varios *workers*)
- Descripción: extensión de PostgreSQL que fragmenta tablas entre nodos; tablas distribuidas, tablas de referencia replicadas en todos los nodos, colocación de tablas por la misma clave, fragmentación por esquema, ejecución paralela en los *workers*
- Cargar Stackoverflow eligiendo la columna de distribución (p. ej. `OwnerUserId` o el `Id` de la pregunta) y qué tablas son de referencia; justificar la elección con las consultas, como en la sesión de Cassandra
- Ejecutar RQ1–RQ4 y mostrar con `EXPLAIN` qué se ejecuta en cada *worker* y qué se reparte por la red; comparar una distribución en la que RQ4 se resuelve localmente con otra en la que no
- Añadir un nodo y rebalancear los fragmentos
- Comparar con CockroachDB y TiDB: fragmentación añadida a PostgreSQL frente a un motor distribuido desde cero (transacciones entre fragmentos, consistencia, compatibilidad con PostgreSQL)
- Artículos: Cubukcu et al., «Citus: Distributed PostgreSQL for Data-Intensive Applications», SIGMOD 2021
- Fuentes: [conceptos de Citus](https://docs.citusdata.com/en/stable/get_started/concepts.html); [Citus 12: fragmentación por esquema](https://www.citusdata.com/blog/2023/07/18/citus-12-schema-based-sharding-for-postgres/); [Pavlo, «Databases in 2025: A Year in Review»](https://www.cs.cmu.edu/~pavlo/blog/2026/01/2025-databases-retrospective.html)